# FluoVolt experiments of Semenov et al. (2025), on the paper's other deposits

Paper: Semenov, Kim, Silkuniene & Pakhomov (2025), Cell Reports Physical Science 6, 102660,
doi:10.1016/j.xcrp.2025.102660. Claims: `claims/001272/10.1016-j.xcrp.2025.102660.yaml`, the claims
marked `data: elsewhere`.

The paper deposited its raw images in ten dandisets without saying which figure is in which; the
claims file assigns them from their titles. Pinned here, each at its most recent published version
(the paper cites none):

- DANDI:001032, 001033 and 001035 (0.240527.1432): 250-ms sine waves at many frequencies, unmodulated
  and modulated at 20 Hz ("spindle"), FluoVolt, NaCl solution. Figures 1 and 2.
- DANDI:001034 (0.240527.1431) appears to duplicate 001032: its 80 2-kHz unmodulated files have frame
  stacks identical to those of 001032 (`di8_controls.ipynb`; its other files are not compared). It is
  not used.
- DANDI:001030 (0.240520.1924): 2-s ramped 2-kHz sine waves in sodium gluconate solution. Figure 3C.
- DANDI:001271 (0.250805.1352): tetrodotoxin and vehicle. Figure S2.

| Claim | Data | Here |
|---|---|---|
| fig1-pure-sine-excites-all | 001032, 001033, 001035 | tested |
| fig1-20khz-needs-twice-the-field | 001032, 001033, 001035 | tested |
| fig2d-modulated-thresholds-higher | 001032, 001033, 001035 | tested |
| fig2-modulated-locks-to-beats | 001032, 001033, 001035 | tested |
| fig2c-pure-more-aps | 001032, 001033, 001035 | tested |
| fig3c-sustained-firing-2s | 001030 | tested |
| figs2-ttx-blocks-depolarization | 001271 | tested |
| fig3b-low-conductivity-halves-threshold | none | blocked: no deposit holds 250-ms sessions in sodium gluconate |
| fig4-pooled-time-constants, fig4-orientation-and-poles | 001273 | `strobe_check.ipynb` |
| fig6-sine-less-than-pulse, threshold-depolarization-estimate | 001274 | blocked (see the attempt record) |

Methods are those of the claims file's header (F1 to F6, O1 to O6), applied to FluoVolt traces
without inversion; `sinewave.py` says where each parameter comes from.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import sinewave as sw

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 300)
print("pinned:", {k: v for k, v in sw.DANDISETS.items() if k != "001272"})

In [ ]:
def fig12_assets(ds):
    return [a for a in sw.list_assets(ds)
            if any(f"_ses-{f}-{kind}-stimulation" in a["path"] for f in ("2kHz", "20kHz") for kind in ("sine", "spindle"))]

FV = []
for ds in ("001032", "001033", "001035"):
    for r in sw.load_prepared(fig12_assets(ds)):
        r["dataset"] = ds
        r["subject"] = f"{ds}:{r['subject']}"
        FV.append(r)
T71 = sw.load_prepared(sw.list_assets("001271"))
L30 = sw.load_prepared(sw.list_assets("001030"), onset_ms=25.0)
print(len(FV), "sessions from 001032/33/35;", len(T71), "from 001271;", len(L30), "from 001030")

## Data check

In [ ]:
fv = pd.DataFrame([dict(dataset=r["dataset"], subject=r["subject"], freq=r["freq_hz"], modulated=r["modulated"],
                        volts=r["volts"], pp=r["peak_to_peak"], naglu=r["nagluconate"], n_frames=r["n_frames"],
                        rate=r["rate"], md5=r["data_md5"], start=r["start"], eroded_empty=r["eroded_empty"]) for r in FV])
print("001032/33/35 sessions by dataset, carrier and modulation:")
print(fv.groupby(["dataset", "freq", "modulated"]).agg(sessions=("subject", "size"), neurons=("subject", "nunique"),
                                                       v_min=("volts", "min"), v_max=("volts", "max")).to_string())
print("frame rates:", sorted(fv.rate.unique()), "| frames per file:", dict(fv.n_frames.value_counts()),
      "| amplitude stated peak-to-peak:", int(fv.pp.sum()), "of", len(fv), "| sodium gluconate:", int(fv.naglu.sum()))
print("identical frame stacks:", int(fv.md5.duplicated().sum()), "| O1 (b) erosion empty in", int(fv.eroded_empty.sum()), "files")
print(f"files of 900 frames cover {898 / 3086.4 * 1000 - 50:.0f} ms of the 250-ms stimulus")
t71 = pd.DataFrame([dict(subject=r["subject"], ttx=r["ttx"], sham=r["sham"], freq=r["freq_hz"], volts=r["volts"],
                         n_frames=r["n_frames"], start=r["start"]) for r in T71])
print(f"\n001271: {len(t71)} sessions, {t71.subject.nunique()} neurons, {t71[t71.ttx].subject.nunique()} with TTX sessions, "
      f"{int(t71.ttx.sum())} TTX sessions, {int(t71.sham.sum())} sham; frequencies {sorted(t71.freq.dropna().unique())}")
l30 = pd.DataFrame([dict(subject=r["subject"], volts=r["volts"], naglu=r["nagluconate"], n_frames=r["n_frames"],
                         rate=r["rate"], start=r["start"]) for r in L30])
l30["neuron"] = l30.subject.str.split("_").str[0]
print(f"\n001030: {len(l30)} sessions; neurons {sorted(l30.neuron.unique())}; volts {sorted(l30.volts.unique())} "
      f"(stated as one polarity); sodium gluconate {int(l30.naglu.sum())}; frames {dict(l30.n_frames.value_counts())} "
      f"at {sorted(l30.rate.unique())} frames/s; the description puts the onset at 25 ms and the end at 2,025 ms")
print(f"6000-frame files end at {5998 / 3134.8 * 1000:.0f} ms, before the stimulus ends")
ttx_desc = [r["description"] for r in T71 if r["ttx"]]
said = sum("caused sustained depolarization" in x for x in ttx_desc)
print(f"001271 TTX sessions whose description says TTX 'caused sustained depolarization': {said} of {len(ttx_desc)}")
if said:
    x = ttx_desc[0]; i = x.find("caused sustained")
    print("  e.g.:", x[max(0, i - 160):i + 90])
print("sodium gluconate in the descriptions: 001032/33/35", int(fv.naglu.sum()), "of", len(fv),
      "| 001271", sum(r["nagluconate"] for r in T71), "of", len(T71))

In [ ]:
# Clock check: the jump of dF at candidate onsets, all 001032/33/35 sessions (true onset 50 ms).
def onset_contrast(recs, onset_ms, roi="whole"):
    vals = []
    for r in recs:
        y = sw.dff(r["traces"][roi], r["t"], "a")
        t = r["t"]
        vals.append(y[(t >= onset_ms) & (t < onset_ms + 20)].mean() - y[(t >= onset_ms - 20) & (t < onset_ms)].mean())
    return np.array(vals)
shifts = [25, 30, 40, 50, 60, 70, 100, 150, 200]
print(pd.DataFrame({f"{s} ms": [onset_contrast(FV, s).mean(), onset_contrast(T71, s).mean()] for s in shifts},
                   index=["001032/33/35", "001271"]).round(3).to_string())
c = onset_contrast(L30, 25.0)
print("001030, onset 25 ms: mean contrast", round(c.mean(), 3), "; at 100 ms:", round(onset_contrast(L30, 100.0).mean(), 3))

# Null for the AP detector: sham sessions (no stimulus) of 001271 and the long sham of 001035.
sham = [r for r in T71 if r["sham"]]
sham_tab = {k: sw.session_table(sham, *k, invert=False) for k in sw.READINGS}
print("\nAPs detected in the", len(sham), "sham sessions of 001271 (250-ms window from 50 ms), per reading:",
      {"/".join(k): int(v.n_aps.sum()) for k, v in sham_tab.items()})

## Session tables under every reading

All 12 combinations of O1 (ROI a, b, whole frame), O2 (F0 a, b) and O3 (height a, b), with the
claims file's AP width of 1 to 20 ms at half height. The claims file had 2 to 20 ms when this attempt
was first run; a reader amended it after the question in `questions/001272.yaml`, since FluoVolt APs
are about 2 ms wide at half height (the widths are printed below) and the Methods' "lasted about
10 ms" is the whole AP duration. The earlier 2-ms bound is run as a variant on every claim, to show
what the change did.

In [ ]:
def rname(k):
    return "/".join(k)

TAB = {k: sw.session_table(FV, *k, invert=False) for k in sw.READINGS}
TAB2 = {k: sw.session_table(FV, *k, invert=False, min_width_ms=2.0) for k in sw.READINGS}
TABP = {k: sw.session_table(FV, *k, invert=False, height_ref="prominence") for k in sw.READINGS}
k = ("whole", "a", "b")
w = []
for r in FV[:200]:
    y = sw.dff(r["traces"]["whole"], r["t"], "a")
    w += list(sw.detect_aps(y, r["t"], "b", min_width_ms=0.3)["widths"])
w = np.array(w)
print(f"AP widths at half height, whole/a/b, lower bound 0.3 ms, first 200 sessions: n = {len(w)}, "
      f"median {np.median(w):.2f} ms, {np.mean(w < 2):.0%} under 2 ms, {np.mean(w < 1):.0%} under 1 ms")
print("APs per reading (1 ms | 2 ms):", {rname(k): (int(TAB[k].n_aps.sum()), int(TAB2[k].n_aps.sum())) for k in sw.READINGS})

# Null test of the detector (added after verification round 1): detections before the onset and from
# 10 ms after the stimulus to the end of the recording, where no AP is expected, against those during
# the stimulus, per 100 ms, with the AP height as the dF value and as prominence.
rows = []
for href, T in (("value", TAB), ("prominence", TABP)):
    for k in sw.READINGS:
        t = T[k]
        post = t[t.post_ms > 0]
        rows.append(dict(reading=rname(k), height_as=href, during=int(t.n_aps.sum()),
                         during_per_100ms=100 * t.n_aps.sum() / t.stim_ms.sum(),
                         before_per_100ms=100 * t.null_pre.sum() / (sw.ONSET_MS * len(t)),
                         after=int(post.null_post.sum()), after_per_100ms=100 * post.null_post.sum() / post.post_ms.sum(),
                         sessions_after=len(post), sham_aps=int(sham_tab[k].n_aps.sum()) if href == "value" else
                         int(sw.session_table(sham, *k, invert=False, height_ref="prominence").n_aps.sum())))
FNULL = pd.DataFrame(rows)
print(FNULL.round(2).to_string(index=False))

The null test: under the whole frame, with the height as the dF value (the claims file's reading,
adopted here), the detector finds 0.09 to 0.10 peaks per 100 ms after the stimulus, against 1.77 to
1.83 per 100 ms during it, and none before the onset or in the 5 sham sessions. With the height as
prominence the whole frame gives 0.02 to 0.05 after the stimulus, and 1 or 2 detections in the sham
sessions. The Otsu ROI with the 5% height finds after the stimulus 1.04 to 1.06 per 100 ms under ROI (a)
(1.07 to 1.10 before the onset), against 2.44 to 2.47 during it, about 43% of that rate, and 2.12 to
2.14 under ROI (b) (2.35 to 2.40 before), against 3.33 to 3.38 during it, so under ROI (b) most of its
detections are noise and under ROI (a) a large part. For
these FluoVolt recordings the value reading with the whole frame is adopted; prominence is shown as a
variant.

In [ ]:
# Traces of one neuron at 2 and 20 kHz, unmodulated and modulated, whole frame, APs of height (b).
subj = fv.groupby("subject").filter(lambda g: g.modulated.any() and (g.freq == 20000).any()).subject.iloc[0]
rs = sorted([r for r in FV if r["subject"] == subj], key=lambda r: (r["freq_hz"], r["modulated"], r["volts"]))
fig, axes = plt.subplots(len(rs), 1, figsize=(12, 0.85 * len(rs)), sharex=True)
lo, hi = np.inf, -np.inf
for ax, r in zip(axes, rs):
    y = sw.dff(r["traces"]["whole"], r["t"], "a")
    ap = sw.detect_aps(y, r["t"], "b")
    ax.plot(r["t"], y, lw=0.5, c="k"); ax.plot(ap["times"] + 50, ap["heights"], "v", c="tab:red", ms=3)
    ax.axvspan(50, 300, color="tab:blue", alpha=0.08); ax.set_ylim(-5, 20)
    ax.text(1, 13, f"{r['freq_hz'] / 1000:g} kHz {'mod' if r['modulated'] else 'pure'} {r['volts']} V", fontsize=7)
    lo, hi = min(lo, y.min()), max(hi, y.max())
axes[0].set_title(subj); axes[-1].set_xlabel("time (ms)"); plt.tight_layout(); plt.show()
print(f"{subj}: plotted dF {lo:.1f} to {hi:.1f} %, axes -5 to 20 %")

## fig1-pure-sine-excites-all

> Unmodulated 250-ms sine waves at 2 and 20 kHz excited every neuron tested, with APs 100 to 200 ms
> after onset near threshold, firing rate rising to a ceiling of 60 to 90 Hz, and an excitation block
> with sustained depolarization at the highest fields.

Criterion: under every reading, at least 90% of neurons excited at each frequency, the median ceiling
(highest F5 rate per neuron, median over neurons) within 54 to 99 Hz, and the median first-AP latency
at each neuron's lowest exciting field within 90 to 220 ms; blocked as methods_underspecified when
the readings disagree. The block is reported under O6, not judged.

In [ ]:
def fig1(tab):
    u = tab[~tab.modulated]
    out = {}
    for f in (2000, 20000):
        g = u[u.freq_hz == f]
        exc = g.groupby("subject").n_aps.max() > 0
        out[f"excited_{f // 1000}k"] = f"{int(exc.sum())}/{len(exc)}"
        out[f"frac_{f // 1000}k"] = exc.mean()
    lat, latf = [], {}
    for f in (2000, 20000):
        g = u[u.freq_hz == f]
        latf[f] = []
        for s, t0 in zip(*[sw.thresholds(g)[c] for c in ("subject", "threshold")]):
            if not np.isnan(t0):
                latf[f].append(np.nanmedian(g[(g.subject == s) & (g.field == t0)].first_ms))
        lat += latf[f]
    ceil = u.groupby("subject").rate_hz.max()
    top = u[u.field == u.groupby("subject").field.transform("max")]
    out.update(ceiling_median=np.nanmedian(ceil), ceiling_min=np.nanmin(ceil), ceiling_max=np.nanmax(ceil),
               latency_median=np.median(lat), latency_q1=np.percentile(lat, 25), latency_q3=np.percentile(lat, 75),
               latency_2k=np.median(latf[2000]), latency_20k=np.median(latf[20000]),
               block_neurons=int(top.groupby("subject").sust_depol_a.any().sum()), neurons=u.subject.nunique())
    out["reproduced"] = (out["frac_2k"] >= 0.9 and out["frac_20k"] >= 0.9 and 54 <= out["ceiling_median"] <= 99
                         and 90 <= out["latency_median"] <= 220)
    return out

full_paths = {r["path"] for r in FV if r["n_frames"] >= 1200}
TABF = {k: t[t.path.isin(full_paths)] for k, t in TAB.items()}
print(f"sessions of 1,200 frames: {len(full_paths)} of {len(FV)}")
for name, T in (("width 1-20 ms", TAB), ("variant, earlier bound, width 2-20 ms", TAB2),
                ("variant, height as prominence", TABP), ("variant, 1,200-frame sessions only", TABF)):
    c1 = pd.DataFrame([dict(reading=rname(k), **fig1(T[k])) for k in sw.READINGS])
    print(name); print(c1.round(2).to_string(index=False))
    print(f"reproduced under {int(c1.reproduced.sum())} of {len(c1)} readings\n")

**Result: not reproduced.** No reading meets the criterion, with the 1-ms width, the earlier 2-ms
bound, the height as prominence, or on the 1,200-frame sessions only, so the readings agree and the
claim is not blocked. Excitation is close to the claim under the whole-frame ROI (27 of 28 neurons at
2 kHz, 15 or 16 of 17 at 20 kHz; 8 to 17 of 17 at 20 kHz over all readings; 28 of 28 and 16 or 17 of 17
with the height as prominence). Each reading fails one of the other two parts. The median over neurons of the highest
first-two-AP rate is 25 to 177 Hz over readings (whole frame 45 to 47 Hz, below the 54-Hz bound; 39 to
43 Hz as prominence), and the median first-AP latency at the lowest exciting field is 15 to 77 ms
(whole frame 62 to 64 ms), below the 90-ms bound. Split by carrier, as the criterion's pooled median
does not, the whole-frame latency is 49 ms at 2 kHz and 112 to 124 ms at 20 kHz. Leaving out the 38
sessions of 900 frames, which end 241 ms into the stimulus, gives whole-frame rates of 47 to 49 Hz and
latencies of 56 to 59 ms. Under a/a/a and a/b/a the rate, 63 Hz, is within its bounds and the latency,
32 and 36 ms, is not; these Otsu-ROI readings with the 5% height detect APs in sham sessions and
before the onset (null test). At most 1 of 32 neurons shows sustained depolarization at its highest
field. With the 2-ms bound the rate was 21 to 61 Hz and the latency 66 to 79 ms.

## fig1-20khz-needs-twice-the-field

> Responses to 20 kHz needed about 2-fold stronger fields than at 2 kHz, and all thresholds at 20 kHz
> were about 2-fold higher than at 2 kHz.

Criterion: thresholds for at least 1 and at least 5 APs (F4), per stimulus (unmodulated, modulated);
each ratio of mean 20-kHz to mean 2-kHz threshold within 1.8 to 2.2, and 20 against 2 kHz at
p < 0.0001 under (a) one test per stimulus and AP count, (b) one test pooling the four; under every
reading, blocked as methods_underspecified when they disagree.

In [ ]:
def thr_groups(tab):
    out = {}
    for mod in (False, True):
        for n in (1, 5):
            for f in (2000, 20000):
                out[(mod, n, f)] = sw.thresholds(tab[(tab.modulated == mod) & (tab.freq_hz == f)], n).dropna(subset=["threshold"])
    return out

def fig1b(tab):
    g = thr_groups(tab)
    rows, ok = [], True
    p20, p2 = [], []
    for mod in (False, True):
        for n in (1, 5):
            a, b = g[(mod, n, 2000)].threshold, g[(mod, n, 20000)].threshold
            ratio = b.mean() / a.mean() if len(b) else np.nan
            p = stats.ttest_ind(b, a).pvalue if len(b) > 1 and len(a) > 1 else np.nan
            ok &= bool(1.8 <= ratio <= 2.2) and bool(p < 1e-4)
            rows.append(f"{'mod' if mod else 'pure'}{n}: {len(a)}/{len(b)} {ratio:.2f} p={p:.1g}")
            p20 += list(b); p2 += list(a)
    pp = stats.ttest_ind(p20, p2).pvalue
    ratio_ok = all(1.8 <= float(r.split()[2]) <= 2.2 for r in rows if r.split()[2] != "nan")
    return dict(groups=" | ".join(rows), pooled_p=pp, reading_a=ok, reading_b=ratio_ok and pp < 1e-4)

for name, T in (("width 1-20 ms", TAB), ("variant, earlier bound, width 2-20 ms", TAB2)):
    c2 = pd.DataFrame([dict(reading=rname(k), **fig1b(T[k])) for k in sw.READINGS])
    print(name, "(group: n 2 kHz / n 20 kHz, ratio, p)"); print(c2.to_string(index=False))
    print(f"reproduced under (a) {int(c2.reading_a.sum())}, under (b) {int(c2.reading_b.sum())} of {len(c2)} readings\n")
k = ("whole", "a", "b")
g = thr_groups(TAB[k])
print("mean thresholds (V/cm, files' 36 V/cm per V) under whole/a/b:",
      {f"{'mod' if m else 'pure'} {n}AP {f // 1000}k": round(v.threshold.mean(), 1) for (m, n, f), v in g.items() if len(v)})

**Result: not reproduced.** No reading meets the criterion under (a) or (b), with either width bound,
so the readings agree. The 20-kHz thresholds are higher than the 2-kHz ones (pooled p at most 1.4e-9;
unmodulated groups p at most 0.003, and below 0.0001 under the whole frame), but in every reading at
least one ratio is outside 1.8 to 2.2. The ratios range from 1.71 to 2.98 (whole frame: 2.27 to 2.39
unmodulated 1 AP, 2.41 to 2.44 for 5 APs, 2.62 modulated 1 AP, 2.82 to 2.83 modulated 5 APs). Under
a/a/a the unmodulated ratios, 2.00 and 2.13, are within the range and the modulated ones, 2.77 and
2.98, are not. The modulated 20-kHz groups hold 1 to 3 neurons. With the 2-ms bound the ratios ranged
from 1.88 to 2.86. The ratios use the files' nominal volts, which the files say are not the delivered
amplitudes, with a frequency-dependent calibration that is not deposited.

## fig2d-modulated-thresholds-higher

> Thresholds for sine waves modulated at 20 Hz were significantly higher than for pure sine waves at
> the same carrier, for 5 APs at 2 kHz and for 1 and 5 APs at 20 kHz, so modulation did not lower
> excitation thresholds.

Criterion: two-tailed unpaired t tests; reproduced when the three modulated thresholds are higher with
p < 0.05 and the 1-AP threshold at 2 kHz is not lower for modulated waves at p < 0.05, under every AP
reading; blocked as methods_underspecified when they disagree.

In [ ]:
def fig2d(tab):
    g = thr_groups(tab)
    res, ok = {}, True
    for f, n, need_higher in ((2000, 5, True), (20000, 1, True), (20000, 5, True), (2000, 1, False)):
        m, u = g[(True, n, f)].threshold, g[(False, n, f)].threshold
        p = stats.ttest_ind(m, u).pvalue if len(m) > 1 and len(u) > 1 else np.nan
        d = m.mean() - u.mean()
        res[f"{f // 1000}k {n}AP"] = f"mod {m.mean():.0f} (n={len(m)}) vs pure {u.mean():.0f} (n={len(u)}), p={p:.4f}"
        ok &= (d > 0 and p < 0.05) if need_higher else not (d < 0 and p < 0.05)
    res["reproduced"] = bool(ok)
    return res

for name, T in (("width 1-20 ms", TAB), ("variant, earlier bound, width 2-20 ms", TAB2)):
    c3 = pd.DataFrame([dict(reading=rname(k), **fig2d(T[k])) for k in sw.READINGS])
    print(name); print(c3.to_string(index=False))
    print(f"reproduced under {int(c3.reproduced.sum())} of {len(c3)} readings\n")

**Result: not reproduced.** No reading meets the criterion, with either width bound, so the readings
agree. Modulated thresholds are higher than unmodulated ones in every group with data, but the 2-kHz
5-AP difference is not significant under any reading (p from 0.0500 to 0.93; 0.085 to 0.48 with the
2-ms bound). Under whole/a/a and whole/b/a the other three parts hold and this p, 0.0500, is at the
bound and not below it. The 20-kHz modulated groups hold 1 to 3 neurons, so their tests are
significant under some readings only (1 AP p from 0.0005 to 0.87; 5 APs p from 0.0096 to 0.058 where
computable). The 2-kHz 1-AP threshold is not lower for modulated waves (higher, p from 0.023 to 0.75),
as the claim requires.

## fig2-modulated-locks-to-beats

> With 20-Hz modulation, firing above threshold locked to the beats, one AP near the peak of each of
> the five beats and at most two per beat (ten in all), and modulated sine waves triggered no
> sustained firing or sustained depolarization.

Criterion: modulated sessions only; each beat peaks 25 ms into its 50 ms; reproduced when no session
has more than 10 APs, in sessions with 5 APs or more most APs fall within 12.5 to 37.5 ms of each beat,
and no session shows sustained depolarization (O6). The phase histogram is reported.

In [ ]:
def fig2b(tab):
    m = tab[tab.modulated]
    ph = np.concatenate([np.mod(t, 50.0) for t in m.ap_times]) if len(m) else np.array([])
    big = m[m.n_aps >= 5]
    frac = [np.mean((np.mod(t, 50) >= 12.5) & (np.mod(t, 50) <= 37.5)) for t in big.ap_times]
    sd_a, sd_b = int(m.sust_depol_a.sum()), int(m.sust_depol_b.sum())
    ok = (m.n_aps.max() <= 10) and (np.mean(np.array(frac) > 0.5) > 0.5 if len(frac) else False) and sd_a == 0 and sd_b == 0
    return dict(sessions=len(m), max_aps=int(m.n_aps.max()), over_10=int((m.n_aps > 10).sum()), sessions_5plus=len(big),
                sessions_5plus_mostly_in_window=int(np.sum(np.array(frac) > 0.5)),
                all_aps_in_window=round(float(np.mean((ph >= 12.5) & (ph <= 37.5))), 2) if len(ph) else np.nan,
                sust_depol_a=sd_a, sust_depol_b=sd_b, sust_firing=int(m.sust_firing.sum()), reproduced=bool(ok))

for name, T in (("width 1-20 ms", TAB), ("variant, earlier bound, width 2-20 ms", TAB2), ("variant, height as prominence", TABP)):
    c4 = pd.DataFrame([dict(reading=rname(k), **fig2b(T[k])) for k in sw.READINGS])
    print(name); print(c4.to_string(index=False))
    print(f"reproduced under {int(c4.reproduced.sum())} of {len(c4)} readings\n")
k = ("whole", "a", "b")
m = TAB[k][TAB[k].modulated]
ph = np.concatenate([np.mod(t, 50.0) for t in m.ap_times])
h, e = np.histogram(ph, bins=10, range=(0, 50))
print("AP phase within the 50-ms beat, whole/a/b, 5-ms bins from 0 ms:", h.tolist(), f"(n = {len(ph)})")
u = TAB[k][~TAB[k].modulated]
phu = np.concatenate([np.mod(t, 50.0) for t in u.ap_times])
print("same for unmodulated sessions (reference):", np.histogram(phu, bins=10, range=(0, 50))[0].tolist())
flag = m[m.sust_depol_a | m.sust_depol_b | (m.n_aps > 10)]
print("modulated sessions with more than 10 APs or sustained depolarization under whole/a/b:")
print(flag[["subject", "freq_hz", "volts", "n_aps", "late_aps", "late_mean", "pre_sd", "level_b"]].round(2).to_string(index=False))
for t in flag.ap_times:
    print("AP times (ms from onset):", np.round(t, 1).tolist())
same = TAB[k][(TAB[k].subject.isin(flag.subject)) & (TAB[k].freq_hz.isin(flag.freq_hz))]
print("all sessions of that neuron at that carrier:")
print(same[["modulated", "freq_hz", "volts", "n_aps", "late_aps", "late_mean", "pre_sd"]].sort_values(["modulated", "volts"]).round(2).to_string(index=False))

**Result: partially reproduced, resting on an assumption.** The locking holds under every reading:
in most sessions with 5 or more APs, most of them fall in the middle half of the beat (whole frame 22
of 25 or 23 of 26 sessions; 66 to 100% of all APs there over the readings, 89 to 91% under the whole
frame), and the whole-frame phase histogram peaks 20 to 35 ms into the 50-ms beat while that of
unmodulated sessions is flat. With the 1-ms width the criterion is met under 4 of 12 readings, the
Otsu-ROI readings with height (b), against 8 of 12 with the earlier 2-ms bound and 4 of 12 with the
height as prominence. Under the Otsu ROI with the 5% height, 10 to 20 sessions have more than 10 APs
(up to 17); in the null test these readings detect peaks after the stimulus at 1.04 to 1.06 per
100 ms under ROI (a) and 2.12 to 2.14 under ROI (b), against 2.44 to 2.47 and 3.33 to 3.38 during it. Under the 4 whole-frame readings, which this record adopts, one
modulated session (001032 neuron 10_1, 2 kHz, 8 V) has 13 APs, against 17 in the neuron's unmodulated
session at the same field, and no modulated session shows sustained depolarization. With the 2-ms
bound, and with the height as prominence (at most 9 APs), the whole-frame readings fail instead on
one modulated session with sustained depolarization by O6 level (a).

## fig2c-pure-more-aps

> In the same neuron at the same field strength, pure sine waves triggered more APs than modulated
> ones, and the threshold was typically lower for unmodulated signals.

Criterion: pairs are a neuron's modulated and unmodulated sessions at the same carrier and field;
reproduced when the unmodulated count is larger in more pairs than the reverse, and the unmodulated
threshold lower in more neurons than the reverse, under every reading. A two-sided paired t test of
the counts is reported.

In [ ]:
def fig2c(tab):
    g = tab.groupby(["subject", "freq_hz", "field", "modulated"]).n_aps.mean().unstack("modulated").dropna()
    g.columns = ["pure", "mod"]
    more, fewer = int((g.pure > g["mod"]).sum()), int((g.pure < g["mod"]).sum())
    p = stats.ttest_rel(g.pure, g["mod"]).pvalue if len(g) > 1 else np.nan
    lower = higher = 0
    for f in (2000, 20000):
        tp = sw.thresholds(tab[(~tab.modulated) & (tab.freq_hz == f)]).set_index("subject").threshold
        tm = sw.thresholds(tab[(tab.modulated) & (tab.freq_hz == f)]).set_index("subject").threshold
        both = pd.concat([tp, tm], axis=1, keys=["pure", "mod"]).dropna()
        lower += int((both.pure < both["mod"]).sum()); higher += int((both.pure > both["mod"]).sum())
    return dict(pairs=len(g), pure_more=more, pure_fewer=fewer, mean_diff=(g.pure - g["mod"]).mean(), paired_p=p,
                thr_pure_lower=lower, thr_pure_higher=higher, reproduced=more > fewer and lower > higher)

for name, T in (("width 1-20 ms", TAB), ("variant, earlier bound, width 2-20 ms", TAB2), ("variant, height as prominence", TABP)):
    c5 = pd.DataFrame([dict(reading=rname(k), **fig2c(T[k])) for k in sw.READINGS])
    print(name); print(c5.round(4).to_string(index=False))
    print(f"reproduced under {int(c5.reproduced.sum())} of {len(c5)} readings\n")

**Result: reproduced.** Under all 12 readings, with either width bound and with the height as
prominence, unmodulated waves evoke more APs than modulated ones at the same neuron, carrier and field
in more of the 52 pairs than the reverse (22 to 47 against 0 to 12 with the 1-ms width; paired t test p
from below 0.0001 to 0.022), and the unmodulated threshold is lower in more neurons than the reverse
(9 to 14 against 0 to 2).

## fig3c-sustained-firing-2s

> In the low-conductivity solution, a 2-s unmodulated 2-kHz sine wave ramped over 250 ms evoked firing
> for its whole duration, rising with field until replaced by a firing block with sustained
> depolarization.

Criterion: traces corrected for bleaching by subtracting a trace without APs (Methods); reproduced
when an intermediate field has APs in the last 250 ms of the stimulus, the AP count rises with field
up to some field, and the highest field shows sustained depolarization (O6, over the last 250 ms).
The onset (25 ms) and the 2-s length are the files' own. The Methods do not say which trace without
APs was subtracted; two readings are run: (own) the neuron's own lowest-field session when it has no
AP, else no correction; (shared) the dataset's AP-free 2-s session (only one exists) subtracted from
every session but itself. The criterion is judged per neuron; a neuron meets it when all three parts hold, and the
claim is reproduced when most neurons meet it (a standard choice).

In [ ]:
def fig3c(k, min_width_ms=1.0, corr="own", href="value", return_table=False):
    roi, f0, h = k
    base = {}
    rows = []
    neurons = sorted({r["subject"].split("_")[0] for r in L30})
    free = {}
    for r in L30:
        y0 = sw.dff(r["traces"][roi], r["t"], f0, onset_ms=25.0)
        ap0 = sw.detect_aps(y0, r["t"], h, stim_ms=2000.0, onset_ms=25.0, min_width_ms=min_width_ms, height_ref=href)
        if len(ap0["times"]) == 0:
            free[r["path"]] = y0
    for n in neurons:
        rs = sorted([r for r in L30 if r["subject"].split("_")[0] == n], key=lambda r: r["volts"])
        for r in rs:
            if corr == "own":
                base[r["path"]] = free.get(rs[0]["path"])
            else:
                other = [p for p in free if p != r["path"]]
                base[r["path"]] = free[other[0]] if other else None
    tab = sw.session_table(L30, roi, f0, h, invert=False, onset_ms=25.0, stim_ms=2000.0, late_ms=250.0,
                           baseline=base, min_width_ms=min_width_ms, height_ref=href)
    tab["neuron"] = tab.subject.str.split("_").str[0]
    tab["corrected"] = [base[p] is not None for p in tab.path]
    tab["aps_last250"] = tab.late_aps
    if return_table:
        return tab, free
    out = []
    for n, g in tab.groupby("neuron"):
        g = g.sort_values("volts")
        inter = g.iloc[1:-1] if len(g) > 2 else g.iloc[0:0]
        late = bool((inter.aps_last250 > 0).any())
        counts = g.n_aps.values
        rises = bool(len(counts) > 1 and counts[np.argmax(counts)] > counts[0])
        top = g.iloc[-1]
        sd = bool(top.sust_depol_a or top.sust_depol_b)
        out.append(dict(neuron=n, volts=g.volts.tolist(), aps=g.n_aps.tolist(), aps_last250=g.aps_last250.tolist(),
                        late_mean=g.late_mean.round(1).tolist(), top_level_a=round(5 * top.pre_sd, 1),
                        top_level_b=round(top.level_b, 1),
                        stim_ms=g.stim_ms.round(0).tolist(), corrected=bool(g.corrected.all()),
                        top_sd_a=bool(top.sust_depol_a), top_sd_b=bool(top.sust_depol_b),
                        meets=late and rises and sd))
    return pd.DataFrame(out)

rows = []
for k in sw.READINGS:
    for corr in ("own", "shared"):
        for href in ("value", "prominence"):
            for mw in (1.0, 2.0):
                d = fig3c(k, mw, corr, href)
                rows.append(dict(reading=rname(k), correction=corr, height_ref=href, width_lo=mw,
                                 neurons_meeting=int(d.meets.sum()), meeting=" ".join(d[d.meets].neuron),
                                 neurons=len(d), reproduced=d.meets.sum() > len(d) / 2))
c6 = pd.DataFrame(rows)
print(c6.pivot_table(index="reading", columns=["correction", "height_ref", "width_lo"], values="meeting",
                     aggfunc="first").replace("", "-").to_string())
print(c6.groupby(["correction", "height_ref", "width_lo"]).agg(reproduced=("reproduced", "sum"), readings=("reproduced", "size"),
                                                  max_neurons=("neurons_meeting", "max")).to_string())
for corr, href in (("own", "value"), ("shared", "value"), ("shared", "prominence")):
    print(f"\nper neuron, width 1 ms, whole/a/b, correction {corr}, height {href}:")
    print(fig3c(("whole", "a", "b"), 1.0, corr, href).to_string(index=False))

In [ ]:
# Null test of the detector on the 2-s recordings (added after verification round 2): detections in
# the 25 ms before the onset and from 10 ms after the stimulus to the end of the recording (2,035 to
# 2,232 ms, in the 15 files of 7,000 frames), against those during the stimulus, per 100 ms.
rows = []
for k in sw.READINGS:
    for corr in ("own", "shared"):
        for href in ("value", "prominence"):
            t, free = fig3c(k, 1.0, corr, href, return_table=True)
            post = t[t.post_ms > 0]
            rows.append(dict(reading=rname(k), correction=corr, height_as=href,
                             during_per_100ms=100 * t.n_aps.sum() / t.stim_ms.sum(),
                             before=int(t.null_pre.sum()), before_per_100ms=100 * t.null_pre.sum() / (25.0 * len(t)),
                             after=int(post.null_post.sum()), after_per_100ms=100 * post.null_post.sum() / post.post_ms.sum(),
                             sessions_after=len(post), ap_free=" ".join(sorted(p.split("/")[-1].split("_ses")[0] for p in free))))
N3C = pd.DataFrame(rows)
print(N3C.round(2).to_string(index=False))

In [ ]:
# c7 at 2.3 V, whole/a/b, shared correction (the same trace as its own: c7 1.0 V): the late detections
# of the value reading, with their prominences, against the 5-SD threshold and the APs earlier on.
from scipy import signal
t, free = fig3c(("whole", "a", "b"), 1.0, "shared", "value", return_table=True)
r = [r for r in L30 if r["subject"] == "c7_2,3V"][0]
base = free[[p for p in free if p != r["path"]][0]]
y = sw.dff(r["traces"]["whole"], r["t"], "a", onset_ms=25.0)
y = y[:len(base)] - base[:len(y)]
ap = sw.detect_aps(y, r["t"], "b", stim_ms=2000.0, onset_ms=25.0)
pk = np.searchsorted(r["t"] - 25.0, ap["times"])
prom = signal.peak_prominences(y, pk)[0]
late = ap["times"] >= 1750
print(f"threshold (5 SD before onset) {ap['threshold']:.2f} %; {late.sum()} detections in the last 250 ms")
print("late: dF at peak", np.round(ap["heights"][late], 1).tolist(), "| prominence", np.round(prom[late], 1).tolist())
print("earlier (first 2 s - 250 ms): prominence median", round(float(np.median(prom[~late])), 1),
      "range", round(float(prom[~late].min()), 1), "to", round(float(prom[~late].max()), 1), f"(n = {(~late).sum()})")

In [ ]:
# The 2-s traces, whole frame, F0 (a), corrected by each neuron's lowest-field trace where it has no AP.
neurons = sorted({r["subject"].split("_")[0] for r in L30})
fig, axes = plt.subplots(len(L30), 1, figsize=(12, 0.8 * len(L30)), sharex=True)
i, lo, hi = 0, np.inf, -np.inf
for n in neurons:
    rs = sorted([r for r in L30 if r["subject"].split("_")[0] == n], key=lambda r: r["volts"])
    for r in rs:
        y = sw.dff(r["traces"]["whole"], r["t"], "a", onset_ms=25.0)
        axes[i].plot(r["t"], y, lw=0.3, c="k"); axes[i].set_ylim(-10, 25)
        axes[i].text(5, 15, f"{n} {r['volts']} V", fontsize=7); axes[i].axvspan(25, 2025, color="tab:blue", alpha=0.05)
        lo, hi = min(lo, y.min()), max(hi, y.max()); i += 1
axes[-1].set_xlabel("time (ms)"); plt.tight_layout(); plt.show()
print(f"uncorrected dF {lo:.1f} to {hi:.1f} %, axes -10 to 25 %")

**Result: partially reproduced, resting on an assumption.** The figure's pattern is present in neuron
c7. The window is the last 250 ms of the stimulus, as this claims file's criterion gives it. Under the whole-frame ROI, which this record adopts, with the AP height taken as prominence, c7
fires through the 2-s wave at 1.45 to 2.0 V (75 to 91 APs, 8 to 10 in the last 250 ms), and at 2.3 V
fires 19 APs, none in the last 250 ms, with a late dF of 6.0% against 5.4% or less at the lower
fields; it meets all three parts. Under the whole frame the only session without APs is c7's own
1.0-V session, so both corrections subtract the same trace from c7's sessions and are one check for
c7, not two. With the AP height taken as the dF value, 9
detections in the last 250 ms at 2.3 V count as firing and c7 fails. With the 1-ms width no reading has
more than 1 of the 4 neurons meeting the criterion, and none of the Otsu-ROI readings has any; with the
earlier 2-ms bound c8 also met it under the shared correction in some readings (at most 2 of 4). c5 and
c6 were stimulated up to 2.0 and 1.45 V only and keep firing; two c5 files end at 1,913 ms, before the
stimulus ends. The prominence reading is adopted here because the late trace at the highest
field sits on a raised plateau, where a height measured as the dF value counts noise peaks (the
verifier's synthetic test). The null test on these recordings (added after verification round 2)
does not settle it: under the whole frame neither reading detects peaks in the 25 ms before the onset,
and after the stimulus, where the trace is back near baseline, the value reading detects none and
prominence 0 to 7 (up to 0.24 per 100 ms, against 1.85 to 2.60 during the stimulus). The 9 late
detections of the value reading in c7 at 2.3 V have a dF of 7.0 to 8.5% at the peak, on a late mean of
6.0%, and prominences of 1.9 to 4.4%, below the 4.43% threshold; but the earlier detections of that
session have prominences of 1.3 to 19.1% (median 4.0%), so prominence does not set the late ones
apart by size. The outcome rests on the prominence reading, which the figure above (a flat plateau
at 2.3 V) and the synthetic test support, and which no null test on 001030 confirms.

## figs2-ttx-blocks-depolarization

> 1 uM tetrodotoxin abolished both the APs and the sustained depolarization caused by intense
> unmodulated 2-kHz sine waves, while the vehicle did not.

Criterion: per neuron, sessions before and about 5 min after TTX or vehicle at the same field;
reproduced when, in every TTX neuron, APs present before are absent after and sustained
depolarization (O6) present before is absent after, while vehicle neurons keep their response.
TTX neurons are those with sessions the files mark as recorded with tetrodotoxin; "before" are their
other sessions. Vehicle neurons have no such session, and nothing in their files marks when the
vehicle was added; the largest gap between consecutive sessions is taken as the change (a choice
made here, as the sessions run in two series with a gap of minutes).

In [ ]:
def ttx_eval(k, min_width_ms=1.0, href="value"):
    tab = sw.session_table([r for r in T71 if not r["sham"]], *k, invert=False, min_width_ms=min_width_ms, height_ref=href)
    tab["t0"] = pd.to_datetime(tab.start)
    out = []
    for s, g in tab.groupby("subject"):
        g = g.sort_values("t0")
        if g.ttx.any():
            before, after, group = g[~g.ttx], g[g.ttx], "TTX"
            gap = (after.t0.min() - before.t0.max()).total_seconds() / 60
        else:
            d = g.t0.diff().dt.total_seconds()
            cut = int(np.argmax(d.fillna(-1).values))
            before, after, group = g.iloc[:cut], g.iloc[cut:], "vehicle"
            gap = d.iloc[cut] / 60
        fields = sorted(set(before.field) & set(after.field))
        b, a = before[before.field.isin(fields)], after[after.field.isin(fields)]
        sd = lambda x: bool((x.sust_depol_a | x.sust_depol_b).any())
        out.append(dict(subject=s, group=group, gap_min=round(gap, 1), fields=len(fields),
                        aps_before=int(b.n_aps.sum()), aps_after=int(a.n_aps.sum()),
                        sd_before=sd(b), sd_after=sd(a),
                        late_before=round(b.late_mean.mean(), 2) if len(b) else np.nan,
                        late_after=round(a.late_mean.mean(), 2) if len(a) else np.nan))
    d = pd.DataFrame(out)
    t, v = d[(d.group == "TTX") & (d.fields > 0)], d[(d.group == "vehicle") & (d.fields > 0)]
    # The criterion's two parts, each conditional on the effect being present before: APs present
    # before are absent after; sustained depolarization present before is absent after. A TTX neuron
    # with no sustained depolarization before does not test the second part; how many do is printed.
    ttx_ok = bool((((t.aps_before > 0) <= (t.aps_after == 0)) & ((~t.sd_before) | (~t.sd_after))).all())
    veh_ok = bool(((v.aps_before == 0) | (v.aps_after > 0)).all())
    return d, dict(ttx_neurons=len(t), vehicle_neurons=len(v), ttx_aps_abolished=int(((t.aps_before > 0) & (t.aps_after == 0)).sum()),
                   ttx_sd_before=int(t.sd_before.sum()), ttx_sd_abolished=int((t.sd_before & ~t.sd_after).sum()),
                   ttx_sd_after=int(t.sd_after.sum()),
                   vehicle_keep_aps=int(((v.aps_before > 0) & (v.aps_after > 0)).sum()),
                   vehicle_sd_before=int(v.sd_before.sum()), vehicle_sd_after=int(v.sd_after.sum()),
                   reproduced=ttx_ok and veh_ok)

rows = []
for k in sw.READINGS:
    for mw, href in ((1.0, "value"), (2.0, "value"), (1.0, "prominence")):
        rows.append(dict(reading=rname(k), width_lo=mw, height_as=href, **ttx_eval(k, mw, href)[1]))
c7 = pd.DataFrame(rows)
print(c7.to_string(index=False))
print(f"height as prominence: reproduced under {int(c7[c7.height_as == 'prominence'].reproduced.sum())} of 12 readings; "
      f"TTX neurons with sustained depolarization before: {c7[c7.height_as == 'prominence'].ttx_sd_before.min()} to "
      f"{c7[c7.height_as == 'prominence'].ttx_sd_before.max()}")
print(f"reproduced under {int(c7[c7.width_lo == 1].reproduced.sum())} of 12 readings (width 1 ms), "
      f"{int(c7[c7.width_lo == 2].reproduced.sum())} of 12 (earlier 2 ms)")
d, _ = ttx_eval(("whole", "a", "b"))
print("\nper neuron, width 1 ms, whole/a/b (matched fields only):")
print(d.to_string(index=False))
dp, _ = ttx_eval(("whole", "a", "b"), href="prominence")
print("\nsame, height as prominence:")
print(dp.to_string(index=False))
print("late mean dF before TTX above 5%:",
      int((d[d.group == "TTX"].late_before > 5).sum()), "of", int((d.group == "TTX").sum()), "TTX neurons")

**Result: reproduced, resting on an assumption.** Under the 4 whole-frame readings, which this record
adopts, and the Otsu-ROI readings with height (b), with either width bound, APs present before TTX are
absent after in every TTX neuron (whole/a/b: 20 to 94 APs before, 0 after, at 4 to 6 matched fields in
all 6), and every vehicle neuron with matched fields keeps firing (13 of 13 under the whole frame).
The criterion's second part is conditional: sustained depolarization (O6) before TTX is found in only
1 of the 6 TTX neurons (14_20240731, under the whole frame), and it is absent after; the other 5 do not
test it. After TTX the late dF is -0.38 to -0.70% in every TTX neuron, against 0.12 to 3.28% before;
no TTX neuron has a late mean above 5% before. Under the Otsu ROI with the 5% height, 1 to 4 TTX
neurons keep detections after TTX, readings that detect 20 APs in 5 sham sessions. With the height as
prominence, a variant, 3 or 4 of the 6 TTX neurons keep detections after TTX under the whole frame
(1 to 6 each under whole/a/b); that reading also detects 1 or 2 peaks in the sham sessions. The files of the TTX sessions say
that TTX "caused sustained depolarization"; the data show none.

## Summary

| Claim | Outcome |
|---|---|
| fig1-pure-sine-excites-all | not reproduced |
| fig1-20khz-needs-twice-the-field | not reproduced |
| fig2d-modulated-thresholds-higher | not reproduced |
| fig2-modulated-locks-to-beats | partially reproduced (assumption) |
| fig2c-pure-more-aps | reproduced |
| fig3c-sustained-firing-2s | partially reproduced (assumption) |
| figs2-ttx-blocks-depolarization | reproduced (assumption) |
| fig3b-low-conductivity-halves-threshold | blocked |
| fig4-pooled-time-constants, fig4-orientation-and-poles | see `strobe_check.ipynb` |
| fig6-sine-less-than-pulse, threshold-depolarization-estimate | blocked |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's: the soma outline is not deposited, so the whole deposited
frame is adopted as the soma where the ROI readings disagree (the claims file's Otsu ROI keeps one
membrane edge, and with the 5% height it detects APs in sham sessions); AP detection follows the
claims file's readings, with the AP width of 1 to 20 ms it now gives and its earlier 2 to 20 ms as a
variant, and the height measured as prominence as a variant (adopted for Figure 3C, where the late
trace sits on a raised plateau); field strengths use the files' nominal volts and
36 V/cm per volt, which the files say overstate the delivered amplitude by a calibration that is not
deposited, so thresholds in V/cm (about 150 V/cm at 2 kHz) are far above the paper's 57.7 V/cm.

What was found about the deposits: the 2-kHz unmodulated files of 001034 duplicate those of 001032 (80 of 80; the others not compared); the stimulus onset is stored only as text
in the 2023-24 files and not at all in the 2024 ones; amplitudes are stated as peak-to-peak in
001032 to 001035 and as one polarity in 001030 with the same 36 V/cm per volt; the TTX files describe
an effect of TTX the data do not show; no deposit holds the 250-ms sodium gluconate sessions of
Figure 3B.